In [1]:
%pip install -q datasets pandas Sastrawi tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import re
import pandas as pd
from datasets import load_dataset
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from tqdm.auto import tqdm

# Memuat dataset dari Hugging Face
dataset = load_dataset("dipawidia/ecommerce-product-reviews-sentiment")
df = pd.concat([dataset[split].to_pandas() for split in dataset.keys()], ignore_index=True)

print("Dataset berhasil dimuat!")
df.head()

d:\TUGAS\TUGAS ORANG\Fira\Tugas2NLP-MutiaMahfira\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset berhasil dimuat!


,review,translate,sentimen
0,Suara dan mic puas. Hanya saya pesan yg nyala ...,sound and mic satisfied. I only ordered the on...,0
1,Good. Mudah-mudahan awet. Makasih,good. hopefully durable. Thanks,1
2,"kondisi dus robek2, tolong di perhatikan lagi","the condition of the box is torn, please pay a...",0
3,Kualitas produk baik. Harga terjangkau dan pen...,product quality is good. affordable prices and...,0
4,"Overall bagus, tombol berfungsi semuanya, tapi...","overall is good, the buttons all work, but the...",0


In [3]:
print(f"Jumlah data (baris) : {df.shape[0]}")
print(f"Jumlah kolom        : {df.shape[1]}")
print(f"Daftar kolom        : {df.columns.tolist()}\n")

df.info()

# Deteksi kolom label sentimen
label_col = next((c for c in df.columns if c.lower() in ['sentiment', 'sentimen', 'label']), df.columns[-1])

ringkasan_label = pd.DataFrame({
    'Jumlah Data': df[label_col].value_counts(),
    'Persentase (%)': (df[label_col].value_counts(normalize=True) * 100).round(2)
})

print(f"\nDistribusi Label Sentimen ('{label_col}'):")
display(ringkasan_label)

Jumlah data (baris) : 11606
Jumlah kolom        : 3
Daftar kolom        : ['review', 'translate', 'sentimen']

<class 'pandas.DataFrame'>
RangeIndex: 11606 entries, 0 to 11605
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     11606 non-null  str  
 1   translate  11606 non-null  str  
 2   sentimen   11606 non-null  int64
dtypes: int64(1), str(2)
memory usage: 2.6 MB

Distribusi Label Sentimen ('sentimen'):


,Jumlah Data,Persentase (%)
sentimen,,
0,5803,50.0
1,5803,50.0


In [4]:
# Deteksi kolom teks ulasan
text_col = next((c for c in df.columns if c.lower() in ['review', 'reviews', 'ulasan', 'text', 'content']), df.columns[0])
df = df.dropna(subset=[text_col]).reset_index(drop=True)

# 1. CASE FOLDING: Mengubah teks menjadi huruf kecil
df['case_folding'] = df[text_col].astype(str).str.lower()

# 2. CLEANING: Menghapus URL, angka, tanda baca, karakter khusus, & huruf berulang
def clean_text(text):
    text = re.sub(r'http\S+|www\.\S+', ' ', text)   # Hapus URL
    text = re.sub(r'[^a-z\s]', ' ', text)            # Hapus angka, tanda baca, & simbol
    text = re.sub(r'(.)\1{2,}', r'\1', text)         # Ubah 'bagussss' -> 'bagus'
    text = re.sub(r'\s+', ' ', text).strip()         # Rapikan spasi
    return text

df['cleaning'] = df['case_folding'].apply(clean_text)

# 3. TOKENISASI: Memisahkan teks menjadi daftar kata
df['tokenisasi'] = df['cleaning'].str.split()

# 4. STOPWORD REMOVAL: Menghapus kata umum yang tidak penting
stopwords_id = set(StopWordRemoverFactory().get_stop_words())
df['stopword_removal'] = [
    [w for w in tokens if w not in stopwords_id and len(w) > 1]
    for tokens in df['tokenisasi']
]

# 5. STEMMING: Mengubah kata berimbuhan ke kata dasar (dengan cache kata unik)
stemmer = StemmerFactory().create_stemmer()
unique_words = {w for tokens in df['stopword_removal'] for w in tokens}

stem_cache = {}
for w in tqdm(unique_words, desc="Proses Stemming"):
    # Kata < 4 huruf tidak berimbuhan, > 20 huruf biasanya typo gabungan
    stem_cache[w] = stemmer.stem(w) if 4 <= len(w) <= 20 else w

df['stemming'] = [[stem_cache[w] for w in tokens] for tokens in df['stopword_removal']]
df['teks_bersih'] = [' '.join(tokens) for tokens in df['stemming']]

print("Preprocessing selesai!")

Proses Stemming: 100%|██████████| 11750/11750 [12:05<00:00, 16.20it/s]

Preprocessing selesai!


In [5]:
pd.set_option('display.max_colwidth', None)

perbandingan = df[[text_col, 'teks_bersih']].head(10).copy()
perbandingan.columns = ['Teks Sebelum Preprocessing', 'Teks Sesudah Preprocessing']
perbandingan.index = range(1, 11)

display(perbandingan)

,Teks Sebelum Preprocessing,Teks Sesudah Preprocessing
1,Suara dan mic puas. Hanya saya pesan yg nyala dan sudah sy pastikan lg klik yg nyala tp yg datang ga nyala,suara mic puas pesan yg nyala sy pasti lg klik yg nyala tp yg datang ga nyala
2,Good. Mudah-mudahan awet. Makasih,good mudah mudah awet makasih
3,"kondisi dus robek2, tolong di perhatikan lagi",kondisi dus robek perhati
4,Kualitas produk baik. Harga terjangkau dan pengirimannya cepet banget. Udh 3x order disini oke lah tp sayangnya klo buat kado gabisa nitip pake kartu ucapan atau dibungkus pake kertas kado :(,kualitas produk baik harga jangkau kirim cepet banget udh order sini oke lah tp sayang klo buat kado gabisa nitip pake kartu ucap bungkus pake kertas kado
5,"Overall bagus, tombol berfungsi semuanya, tapi packaging kurang karena sudah penyot dan robek sedikit",overall bagus tombol fungsi semua packaging kurang penyot robek sedikit
6,"Produk ini original, Dan kualitasnya sangat baik. kecepatan pengiriman sanagt cepat, recommended seller.",produk original kualitas sangat baik cepat kirim sanagt cepat recommended seller
7,kualitas sangat jelek. bocor,kualitas sangat jelek bocor
8,"terimakasih banyak ka ya, barang sudah diterima dengan baik. sukses terus ka",terimakasih banyak ka barang terima baik sukses terus ka
9,"Body nya lebih berat dari cm 1579, spiral nya lebih tipis dibanding cm 1579, semoga awet, beli CM 1659 penasaran Aja, yang cm 1579 udah mau rusak mesin nya, Harga lebih murah 4k dari cm 1579 Fast delivery, courier Ramah sangat, ty seller",body nya lebih berat cm spiral nya lebih tipis banding cm moga awet beli cm penasaran aja cm udah mau rusak mesin nya harga lebih murah cm fast delivery courier ramah sangat ty seller
10,pesanan di terima tanggal 27 January 2021 tetapi baru bisa saya buka pagi ini tanggal 29 January 2021 pukul 06.00 dan ternyata tab SKETSA ADVAN tidak bisa di nyalakan sama sekali dan di cas juga posisi Tab tidak bisa menyala,pesan terima tanggal january baru buka pagi tanggal january pukul nyata tab sketsa advan nyala sama sekali cas posisi tab nyala


### Pengaruh Preprocessing terhadap Kualitas Data Teks

1. **Mengurangi Noise (Derau Data):** Tahap *cleaning* menghapus URL, angka, tanda baca, dan karakter khusus yang tidak memiliki makna sentimen sehingga data menjadi lebih bersih.
2. **Menyeragamkan Format Kata:** Tahap *case folding* mengubah seluruh huruf menjadi huruf kecil sehingga kata seperti `"Bagus"`, `"BAGUS"`, dan `"bagus"` dikenali sebagai satu entitas yang sama.
3. **Mengurangi Dimensi Fitur:** Tahap *stopword removal* menghapus kata hubung/umum (seperti *yang, dan, di, ke*) yang sering muncul namun tidak membedakan sentimen, sehingga beban komputasi berkurang.
4. **Menyatukan Variasi Kata Berimbuhan:** Tahap *stemming* mengembalikan kata berimbuhan ke kata dasarnya (misalnya *"mengecewakan"*, *"dikecewakan"*, *"kecewa"* menjadi *"kecewa"*), sehingga bobot frekuensi kata menjadi lebih akurat saat pemodelan.